# Advanced Topics

In [0]:
from pyspark.sql.functions import *

In [0]:
txn_data = [
    (1, "IN", 5000),
    (2, "US", 8000),
    (3, "IN", 3000)
]



df_txn = spark.createDataFrame(
    txn_data,
    ["txn_id", "country_code", "amount"]
)

df_txn.display()


txn_id,country_code,amount
1,IN,5000
2,US,8000
3,IN,3000


In [0]:
country_data = [
    ("IN", "India"),
    ("US", "United States")
]

df_country = spark.createDataFrame(
    country_data,
    ["country_code", "country_name"]
)

df_country.display()

country_code,country_name
IN,India
US,United States


## BroadCast Join

In [0]:
# 1st decide which join to pply then optimize using broadcast
df_txn.join(broadcast(df_country),on="country_code",how="inner").display()

country_code,txn_id,amount,country_name
IN,1,5000,India
US,2,8000,United States
IN,3,3000,India


### Skew Handling Using Salting

In [0]:
data = [
    ("Mumbai", 1000),
    ("Mumbai", 2000),
    ("Pune", 1500),
    ("Delhi", 1800)
]

df = spark.createDataFrame(data, ["city", "sales"])
df.display()

city,sales
Mumbai,1000
Mumbai,2000
Pune,1500
Delhi,1800


In [0]:
salted_df = df.withColumn(
    "salted_city",
    concat(
        col("city"),
        lit("_"),
        floor(rand() * 4)
    )
)

salted_df.display()

city,sales,salted_city
Mumbai,1000,Mumbai_3
Mumbai,2000,Mumbai_1
Pune,1500,Pune_3
Delhi,1800,Delhi_2


In [0]:
data = [
    (1, "user1", 100),
    (2, "user1", 200),
    (3, "user1", 300),
    (4, "user2", 150),
    (5, "user3", 400)
]

df = spark.createDataFrame(data, ["txn_id", "user_id", "amount"])
df.show()

+------+-------+------+
|txn_id|user_id|amount|
+------+-------+------+
|     1|  user1|   100|
|     2|  user1|   200|
|     3|  user1|   300|
|     4|  user2|   150|
|     5|  user3|   400|
+------+-------+------+



In [0]:
df_salted = df.withColumn("salt",floor(rand()*4))
df_salted.display()

txn_id,user_id,amount,salt
1,user1,100,1
2,user1,200,3
3,user1,300,1
4,user2,150,0
5,user3,400,1


In [0]:
df_expanded = df.withColumn(
    "salt",
    explode(array([lit(i) for i in range(3)]))
)

df_expanded.display()

txn_id,user_id,amount,salt
1,user1,100,0
1,user1,100,1
1,user1,100,2
2,user1,200,0
2,user1,200,1
2,user1,200,2
3,user1,300,0
3,user1,300,1
3,user1,300,2
4,user2,150,0


In [0]:
result = df_salted.alias("a").join(
    df_expanded.alias("b"),
    ["user_id", "salt"]
)
result.display()


user_id,salt,txn_id,amount,txn_id,amount
user1,1,1,100,3,300
user1,1,1,100,2,200
user1,1,1,100,1,100
user1,1,3,300,3,300
user1,1,3,300,2,200
user1,1,3,300,1,100
user2,0,4,150,4,150
user3,1,5,400,5,400
